# Checking how to enable PyApprox functionality in SPAROW

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyapprox.util.backends.numpy import NumpyBkd
from pyapprox_benchmarks.statest import (
    PolynomialEnsembleBenchmark,
)
from pyapprox.statest.statistics import MultiOutputMean
from pyapprox.statest.mc_estimator import MCEstimator
from pyapprox.statest import (
    MLMCEstimator, MFMCEstimator, GMFEstimator, GRDEstimator, GISEstimator,
)
from pyapprox.statest.acv import ACVAllocator, default_allocator_factory
from pyapprox.statest.acv.base import FittedACVEstimator
from pyapprox.statest.acv.search import ACVSearch
from pyapprox.statest.acv.strategies import (
    FixedRecursionStrategy, TreeDepthRecursionStrategy,
)
from pyapprox.statest.allocation import MCAllocator, CVAllocator
from pyapprox.statest.plotting import (
    plot_allocation, plot_estimator_variance_reductions,
)
from pyapprox.optimization.minimize.scipy.slsqp import ScipySLSQPOptimizer

from sparow.ci.cli import load_problem_adapter, load_scenarios, load_xhat
from sparow.ci.evaluate_true_optimality_gap import TrueOptimalityGapEvaluator
from sparow.ci.pyapprox_interface import (
    convert_pyapprox_allocation_to_acvmrp_params, build_pyapprox_mf_problem_from_adapter, estimate_model_cost
)

[    0.00] Initializing mpi-sppy
Alternative solutions package from or_topas is available.


In [2]:
bkd = NumpyBkd()
np.random.seed(42)

# SLSQP is more robust than the default trust-constr optimizer for ACV allocation
optimizer = ScipySLSQPOptimizer(maxiter=200)
allocator_factory = lambda est: default_allocator_factory(est, optimizer=optimizer)

# ------------------------------------------------------
# User settings
# ------------------------------------------------------
MODEL_MODULE = "sparow_examples.mrp_facilityloc.mrp_discrete_facilityloc"
MODEL_NAME = "HF"
SCENARIO_FILE = "discrete_facilityloc_scenarios.npy"
XHAT_FILE = "../../../sparow/sparow/ci/manually_created_suboptimal_xhat.npy"

# Every replication makes use of BATCH_SIZE iid draws of scenarios
BATCH_SIZE = 100
SOLVER_NAME = "gurobi_direct"
SEED = 678

# ------------------------------------------------------
# Load adapter, scenarios, and candidate solution
# ------------------------------------------------------
problem_adapter = load_problem_adapter(
    model_module_name=MODEL_MODULE,
    model_name=MODEL_NAME,
    use_integer=False,
    lf_model_type="classic",
)

full_scenarios = load_scenarios(SCENARIO_FILE)
xhat = load_xhat(XHAT_FILE)
if "ROOT" in xhat:
    xhat = xhat["ROOT"]
print("Candidate first-stage solution:")
print(xhat)

# ------------------------------------------------------
# Build PyApprox multifidelity problem
# ------------------------------------------------------
problem, bkd = build_pyapprox_mf_problem_from_adapter(
    problem_adapter=problem_adapter,
    full_scenarios=full_scenarios,
    xhat=xhat,
    batch_size=BATCH_SIZE,
    solver_name=SOLVER_NAME,
    solver_options=None,
    seed=SEED,
)

# In this construction:
#   model 0 = HF replication-level gap estimator F_n(\hat{x})
#   model 1 = LF replication-level gap estimator G_n(\hat{x})
# and the PyApprox prior is the empirical distribution of full batches of scenarios.

models = problem.models()
variable = problem.prior()
costs = problem.costs()
nqoi = models[0].nqoi()
nmodels = len(models)

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Candidate first-stage solution:
{'x[0]': 0.0, 'x[1]': 0.0, 'x[2]': 0.0, 'x[3]': 0.0, 'x[4]': 1.0, 'x[5]': 1.0}


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF

In [ ]:
# HACK: Artificially inflate the costs in order to check code correctness
import time

class DelayedModel:
    def __init__(self, model, delay_seconds=0.0):
        self.model = model
        self.delay_seconds = delay_seconds

    def nvars(self):
        return self.model.nvars()

    def nqoi(self):
        return self.model.nqoi()

    def __call__(self, samples):
        if self.delay_seconds > 0:
            time.sleep(self.delay_seconds)
        return self.model(samples)

hf_model = DelayedModel(models[0], delay_seconds=10.0)   # make HF model more expensive
lf_model = DelayedModel(models[1], delay_seconds=0.0)   # no extra LF delay

models = [hf_model, lf_model]

# Recompute costs using the delayed models
hf_cost = estimate_model_cost(hf_model, variable, ntrials=20)
lf_cost = estimate_model_cost(lf_model, variable, ntrials=20)
costs = bkd.array([hf_cost, lf_cost])

In [4]:
# This prints the estimated wall-clock cost of one replication-level
# evaluation of each model. Model 0 is HF, model 1 is LF.
# These costs are the inputs to PyApprox's sample allocation step.
costs_np = bkd.to_numpy(costs)
print(f"{nmodels} models, {nqoi} QoI(s)")
for a, c in enumerate(costs_np):
    print(f"  model {a}: estimated cost = {c:.6f}")

2 models, 1 QoI(s)
  model 0: estimated cost = 0.948646
  model 1: estimated cost = 0.795761


## Stage 1: Pilot Study

Evaluate all models at a shared set of pilot samples to estimate the cross-covariance.

In [ ]:
N_pilot = 10

# Draw N_pilot independent scenario batches from the prior.
# Each batch is one replication in the MRP / ACV-MRP sense.
samples_pilot = variable.rvs(N_pilot)
print(samples_pilot)

# Evaluate every model on the same pilot batches.
vals_pilot = [m(samples_pilot) for m in models]
for val in vals_pilot:
    print(val)
print("\n")

stat = MultiOutputMean(nqoi, bkd)
cov_pilot, = stat.compute_pilot_quantities(vals_pilot)
stat.set_pilot_quantities(cov_pilot)

# Inspect pilot correlations with HF model
# In the 2-model setting, this is the empirical analogue of \rho_{fg}.
cov_np = bkd.to_numpy(cov_pilot)
for a in range(1, nmodels):
    rho = cov_np[0, a] / np.sqrt(cov_np[0, 0] * cov_np[a, a])
    print(f"  Pilot correlation ρ(f0, f{a}) = {rho:.4f}")

In [ ]:
total_budget = 200.0 # Total computational budget is in terms of wall clock time
pilot_cost   = float(costs_np.sum()) * N_pilot
remaining    = total_budget - pilot_cost
print(f"Pilot cost: {pilot_cost:.1f}  |  Remaining: {remaining:.1f}")

## Stage 2: Build Estimator and Allocate

In [ ]:
est = MFMCEstimator(stat, costs)
allocator = default_allocator_factory(est)
result = allocator.allocate(remaining)
fitted = FittedACVEstimator(est, result)

# This prints the PyApprox-recommended number of replication-level evaluations
# for each model under the remaining budget.
print(f"Samples per model: {fitted.nsamples_per_model()}")

# This is PyApprox's predicted standard deviation of the final multifidelity
# based on the pilot covariance estimates and the chosen sample allocation.
print(f"Predicted std:     {float(fitted.covariance()[0,0])**0.5:.6f}")

## Stage 3: Generate Samples

In [ ]:
# This shows the shapes of the actual sampled inputs allocated to each model.
# Each column corresponds to one scenario batch / one replication.
samples_per_model = fitted.generate_samples_per_model(variable.rvs)
print(f"Sample shapes: {[s.shape for s in samples_per_model]}")

## Stage 4: Evaluate Models

In [ ]:
# Evaluate each model on its allocated batches
values_per_model = [models[a](samples_per_model[a]) for a in range(nmodels)]

## Stage 5: Compute Estimate

In [ ]:
# True optimality gap for comparison
problem_adapter.set_active_fidelity("high")

true_gap_evaluator = TrueOptimalityGapEvaluator(
    problem_adapter=problem_adapter,
    scenarios=full_scenarios,
    solver_name=SOLVER_NAME,
)

true_gap_results = true_gap_evaluator.compute_true_gap(xhat=xhat)

true_optimal_value = true_gap_results["true_optimal_value"]
xhat_true_value = true_gap_results["xhat_true_value"]
true_gap = true_gap_results["true_gap"]

print("\nTrue finite-population HF quantities:")
print(f"  True optimal value: {true_optimal_value:.6f}")
print(f"  Candidate true objective: {xhat_true_value:.6f}")
print(f"  True optimality gap: {true_gap:.6f}")
print("\n")

# Estimated optimality gap
estimate = fitted(values_per_model)
estimate_scalar = np.asarray(estimate).item()
print(f"Estimated mean of HF replication output: {estimate_scalar:.6f}")
print(f"True finite-population optimality gap: {true_gap:.6f}")
print(f"Difference: {estimate_scalar - true_gap:.6f}")

## Compare againt MC

In [ ]:
stat_mc = MultiOutputMean(nqoi, bkd)
stat_mc.set_pilot_quantities(cov_pilot[:1, :1])
mc_est = MCEstimator(stat_mc, costs[:1]) # This is HF-only MC Estimator
mc_fitted = MCAllocator(mc_est).allocate(remaining)

mc_var  = float(mc_fitted.covariance()[0, 0])
mf_var  = float(fitted.covariance()[0, 0])
print(f"MC std:  {mc_var**0.5:.6f}")
print(f"MF std:  {mf_var**0.5:.6f}")
print(f"Variance reduction: {mc_var / mf_var:.1f}×")

In [ ]:
# ------------------------------------------------------
# Plot variance reduction
# ------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 3.5))
plot_estimator_variance_reductions([fitted], ["FacilityLoc MFMC"], ax)
ax.set_title("Variance reduction vs MC", fontsize=11)
plt.tight_layout()
plt.show()